# 3.RunnablePassthrough简化invoke调用

知识点讲解：RunnablePassthrough 数据透传与字段追加

1. 什么是 RunnablePassthrough
   - LCEL 两大核心工具类之一（另一个是 RunnableParallel）
   - 最基础的语义：把输入原封不动地传给下游，invoke(x) 直接返回 x
   - 常与 RunnableParallel 配合，用于「获取数据并保持或新增键」
   - 课程文档定位：透传上游参数输入，可以不变或新增额外的键

2. 简化 invoke 调用：裸字符串包装成字典（基础用法）
   - 直接透传用法：{"query": RunnablePassthrough()} 把裸字符串包装成 {"query": 字符串}
   - 这样调用方可以写 chain.invoke("你好，你是") 而不是 chain.invoke({"query": "..."})
   - 课程文档示例输出：你好！是的，我是ChatGPT...
   - 注意：RunnablePassthrough() 获取的是「整个输入」（字符串或字典）

3. RunnablePassthrough 的两种用法
   - 直接透传：RunnablePassthrough()，常用于把裸输入包装成 dict
   - 追加字段：RunnablePassthrough.assign(**kwargs)，保留输入 dict 全部原有键并追加新键
   - 若想从输入字典中取出某一部分（而非整体），用 itemgetter：
       {"query": itemgetter("query")} | prompt | llm | parser

4. assign() 的工作机制（本示例核心）
   - 签名：RunnablePassthrough.assign(**kwargs: Runnable | Callable) -> RunnableAssign
   - 每个 kwarg 的 key 是新增字段名，value 是计算该字段的 Runnable 或函数
   - value 函数接收「完整的原始输入 dict」，返回该字段的值
   - 输出 = 原始输入 dict ∪ 新增字段（即 {**input, **new_fields}）
   - 新增字段之间并行计算，内部同样基于 RunnableParallel 实现

5. assign 与显式并行字典的对比（对照本章第 2 个示例）
   - 显式字典写法：
       {"context": lambda x: retrieval(x["query"]), "query": itemgetter("query")}
     必须手动列出每一个下游需要的字段，query 要写一次透传，字段多时非常冗长
   - assign 写法：
       RunnablePassthrough.assign(context=lambda x: retrieval(x["query"]))
     只声明「新增什么」，原有字段自动保留，代码更短且不易漏字段
   - 二者对 prompt 而言输出完全一致：{"query": ..., "context": ...}

6. 数据流转全过程
   {"query": "你好，我是谁?"}
        ↓ RunnablePassthrough.assign(context=...)
        ↓   （原有 query 保留，并行计算 context = retrieval(query)）
   {"query": "你好，我是谁?", "context": "我是慕小课"}
        ↓ prompt（渲染模板，填充两个变量）
   [HumanMessage(content="请根据用户的问题回答...")]
        ↓ llm
   AIMessage(content="...")
        ↓ parser
   "最终回答字符串"

7. 典型应用场景
   - RAG：在原输入上追加 context 字段，是 LangChain 官方 RAG 模板的标准写法
   - 记忆注入：追加 history 字段，如 assign(history=lambda x: memory.messages)
   - 多字段派生：assign(context=..., timestamp=..., user_profile=...) 一次追加多个
   - 链路中途保留原始输入：便于后续步骤引用最初的 query 做引用溯源

8. 注意事项
   - assign 的输入必须是 dict，传入字符串会报错（需先用 {"key": RunnablePassthrough()} 包装）
   - 若 assign 的字段名与输入已有字段重名，新值会「覆盖」旧值
   - assign 的 value 函数只接受一个参数（完整输入 dict），多参数请用 partial / bind
   - RunnablePassthrough.assign 返回的是 RunnableAssign 实例，而非 RunnablePassthrough


In [ ]:
import dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI
dotenv.load_dotenv()

In [ ]:
def retrieval(query: str) -> str:
    """
    一个模拟的检索器函数
    作用：模拟 RAG 中的检索环节，依据用户问题返回参考上下文
    参数：
        query: str - 用户原始提问文本
    返回：
        str - 检索到的上下文文本（示例中固定返回模拟资料）
    副作用：
        打印检索日志，便于观察 assign 分支的实际执行时机
    """
    print("正在检索:", query)
    return "我是慕小课"

In [ ]:
prompt = ChatPromptTemplate.from_template("""请根据用户的问题回答，可以参考对应的上下文进行生成。
<context>
{context}
</context>
用户的提问是: {query}""")
llm = ChatOpenAI(model="deepseek-flash")
parser = StrOutputParser()
chain = RunnablePassthrough.assign(context=lambda x: retrieval(x["query"])) | prompt | llm | parser
content = chain.invoke({"query": "你好，我是谁?"})
print(content)